# Wake iteration sensitivity example

このNotebookでは、1つのOpenVSPモデルに対してVSPAEROの安定微係数解析を繰り返し実行し、`WakeNumIter`による結果の変化をCSVへ保存する。

処理は上から順に次の流れで進む。

1. リポジトリと入力モデルを確認する。
2. 解析条件と`WakeNumIter`を設定する。
3. 元モデルを一度だけ事前検証する。
4. `WakeNumIter`ごとに独立したcaseディレクトリを作る。
5. VSPAEROを実行し、生成された`.stab`を読む。
6. 安定微係数と実行時間を1行へ整理する。
7. 全caseをCSVへ保存する。

このNotebookはOpenVSP Python APIが使える環境で手動実行する解析例であり、CIで毎回実行する軽量テストではない。

結果の可視化は、生成した`wake_iter_sensitivity.csv`を`plot_wake_iter_sensitivity_refactored.ipynb`で読み込んで行う。


## 1. importとリポジトリ位置

Notebookの実行場所から親ディレクトリを順に調べ、`src/AnalysisVSPAERO.py`を含むリポジトリルートを見つける。


In [1]:
from __future__ import annotations

from pathlib import Path
import math
import shutil
import sys
import time

import pandas as pd
from IPython.display import display

start_dir = Path.cwd().resolve()
repo_root = next(
    (
        candidate
        for candidate in [start_dir, *start_dir.parents]
        if (candidate / "src" / "AnalysisVSPAERO.py").exists()
    ),
    None,
)

if repo_root is None:
    raise FileNotFoundError(
        "Could not find the repository root containing src/AnalysisVSPAERO.py. "
        "Run this notebook inside the repository or set repo_root manually."
    )

if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from src.AnalysisVSPAERO import (  # noqa: E402
    validate_vsp3_for_stability_derivatives,
    vsp_stability_derivatives,
)
from src.TrimTurnSolver import (  # noqa: E402
    read_vspaero_stab,
    resolve_control_columns_from_stab,
)
from src.util import workdir  # noqa: E402

print("repo_root:", repo_root)


repo_root: C:\Users\mtkbirdman\OneDrive\Documents\OpenVSP


## 2. 入力モデルと解析条件

このセルが、通常変更する設定をまとめた場所である。

`wake_iters`は実際に比較したい値を明示的に並べる。未使用の生成規則や、後から上書きされる設定は置かない。


In [2]:
model_name = "G103A"

vsp3_path = (
    repo_root
    / "examples"
    / "models"
    / model_name
    / f"{model_name}.vsp3"
)
output_dir = (
    repo_root
    / "examples"
    / "notebooks"
    / "wake_iter_sensitivity"
)
results_path = output_dir / "wake_iter_sensitivity.csv"

wake_iters = [
    3,
    4,
    8,
    10,
    11,
    13,
    14,
    15,
    16,
    18,
    19,
    20,
]

alpha_deg = 2.0
mach = 0.1
reynolds = 4.4e6
ncpu = 8

fixed_wake_flag = None
redirect_file = ""
verbose = 1
vspaero_verbose = 0

output_dir.mkdir(parents=True, exist_ok=True)

if not vsp3_path.exists():
    raise FileNotFoundError(vsp3_path)

print("vsp3_path:", vsp3_path)
print("output_dir:", output_dir)
print("wake_iters:", wake_iters)


vsp3_path: C:\Users\mtkbirdman\OneDrive\Documents\OpenVSP\examples\models\G103A\G103A.vsp3
output_dir: C:\Users\mtkbirdman\OneDrive\Documents\OpenVSP\examples\notebooks\wake_iter_sensitivity
wake_iters: [3, 4, 8, 10, 11, 13, 14, 15, 16, 18, 19, 20]


## 3. 元モデルの事前検証

各caseへコピーする前に、元の`.vsp3`についてGeom、Set、Control Surface Group、VSPAERO設定を確認する。

全caseで同じモデルを使うため、この検証は通常1回でよい。


In [3]:
validation = validate_vsp3_for_stability_derivatives(
    vsp3_path,
    verbose=2,
)

print("passed:", validation["passed"])
print("errors:", validation["errors"])
print("warnings:", validation["warnings"])

if not validation["passed"]:
    raise RuntimeError(validation["errors"])



-> Validate G103A stability-derivative preflight: C:\Users\mtkbirdman\OneDrive\Documents\OpenVSP\examples\models\G103A\G103A.vsp3
 Checking required Geoms...
 Checking required control-surface Subsurfaces...
 Checking ThickGeom / ThinGeom sets...
 Checking VSPAERO settings...
 Checking VSPAERO control-surface groups and gains...
 Validation PASSED: 0 error(s), 0 warning(s)
passed: True
errors: []
warnings: []


## 4. case準備と`.stab`読込の補助処理

主ループを上から読みやすくするため、次の3つだけを独立した処理として分ける。

- caseディレクトリを作り、古い解析出力を削除して入力`.vsp3`をコピーする。
- caseディレクトリから対応する`.stab`を見つける。
- `.stab`の値をCSVの1行へ展開する。

Control Surface Groupは`.stab`の情報から解決し、`ConGrp_N`だけでなく`delta_a`、`delta_e`、`delta_r`という意味の明確な列名も出力する。


In [4]:
AERO_COEFFICIENTS = [
    "CL",
    "CD",
    "CS",
    "CMl",
    "CMm",
    "CMn",
]

def prepare_case_vsp3(
    case_dir: Path,
    source_vsp3_path: Path,
) -> Path:
    case_dir.mkdir(parents=True, exist_ok=True)
    case_vsp3_path = case_dir / source_vsp3_path.name

    for path in case_dir.glob(f"{case_vsp3_path.stem}*"):
        if path.is_file() and path.suffix.lower() != ".vsp3":
            path.unlink()

    shutil.copy2(source_vsp3_path, case_vsp3_path)
    return case_vsp3_path

def find_stab_file(
    case_dir: Path,
    preferred_stem: str,
) -> Path | None:
    preferred_path = case_dir / f"{preferred_stem}.stab"
    if preferred_path.exists():
        return preferred_path

    candidates = sorted(
        case_dir.glob("*.stab"),
        key=lambda path: path.stat().st_mtime,
        reverse=True,
    )
    return candidates[0] if candidates else None

def flatten_stab_result(stab) -> dict[str, object]:
    control_columns = resolve_control_columns_from_stab(stab)

    row: dict[str, object] = {
        "Sref": float(stab.Sref),
        "Cref": float(stab.Cref),
        "Bref": float(stab.Bref),
        "Vinf": float(stab.V0),
        "rho": math.nan if stab.rho0 is None else float(stab.rho0),
        "AoA_deg": stab.base_condition.get("AoA"),
        "Beta_deg": stab.base_condition.get("Beta"),
        "control_groups": repr(stab.control_groups),
    }

    for delta_name in ("delta_a", "delta_e", "delta_r"):
        row[f"control_column_{delta_name}"] = control_columns.get(
            delta_name,
            "",
        )

    for coefficient_name in AERO_COEFFICIENTS:
        if coefficient_name not in stab.derivatives.index:
            continue

        derivative_row = stab.derivatives.loc[coefficient_name]

        for derivative_name, value in derivative_row.items():
            row[f"{coefficient_name}_{derivative_name}"] = float(value)

        for delta_name, derivative_column in control_columns.items():
            if derivative_column in derivative_row.index:
                row[f"{coefficient_name}_{delta_name}"] = float(
                    derivative_row[derivative_column]
                )

    return row


## 5. Wake iteration sweep

各`WakeNumIter`について、入力モデルのコピー、VSPAERO実行、`.stab`読込、結果整理を順番に行う。

caseごとの失敗は結果行に残し、成功したcaseだけに安定微係数を追加する。


In [ ]:
rows: list[dict[str, object]] = []
sweep_start = time.perf_counter()

for case_index, wake_iter in enumerate(wake_iters, start=1):
    case_name = f"wake_iter_{wake_iter:03d}"
    case_dir = output_dir / case_name
    case_vsp3_path = prepare_case_vsp3(
        case_dir,
        vsp3_path,
    )

    print(
        f"[{case_index}/{len(wake_iters)}] "
        f"WakeNumIter={wake_iter}",
        flush=True,
    )
    case_start = time.perf_counter()

    with workdir(case_dir):
        report = vsp_stability_derivatives(
            case_vsp3_path.name,
            alpha=alpha_deg,
            mach=mach,
            reynolds=reynolds,
            ncpu=ncpu,
            wake_num_iter=wake_iter,
            fixed_wake_flag=fixed_wake_flag,
            redirect_file=redirect_file,
            stop_before_run=False,
            vspaero_verbose=vspaero_verbose,
            verbose=verbose,
        )

    timing = report.get("timing", {})
    stab_path = find_stab_file(
        case_dir,
        case_vsp3_path.stem,
    )

    row: dict[str, object] = {
        "wake_iter": int(wake_iter),
        "passed": bool(report.get("passed", False)),
        "case_dir": str(case_dir),
        "vsp3_path": str(case_vsp3_path),
        "stab_path": "" if stab_path is None else str(stab_path),
        "errors": repr(report.get("errors", [])),
        "warnings": repr(report.get("warnings", [])),
        "elapsed_s": time.perf_counter() - case_start,
        "total_elapsed_s": timing.get("total_elapsed_s", math.nan),
        "compute_geometry_elapsed_s": timing.get(
            "compute_geometry_elapsed_s",
            math.nan,
        ),
        "vspaero_sweep_elapsed_s": timing.get(
            "vspaero_sweep_elapsed_s",
            math.nan,
        ),
        "result_extraction_elapsed_s": timing.get(
            "result_extraction_elapsed_s",
            math.nan,
        ),
    }

    if row["passed"] and stab_path is not None:
        stab = read_vspaero_stab(stab_path)
        row.update(flatten_stab_result(stab))

    rows.append(row)

    print(
        f"    passed={row['passed']}, "
        f"elapsed={row['elapsed_s']:.1f} s, "
        f"stab={row['stab_path']}",
        flush=True,
    )

results = (
    pd.DataFrame(rows)
    .sort_values("wake_iter")
    .reset_index(drop=True)
)
results.to_csv(results_path, index=False)

print(
    f"Done. elapsed="
    f"{time.perf_counter() - sweep_start:.1f} s"
)
print("results_path:", results_path)


[1/12] WakeNumIter=3
    passed=True, elapsed=0.0 s, stab=
[2/12] WakeNumIter=4
    passed=True, elapsed=0.0 s, stab=
[3/12] WakeNumIter=8
    passed=True, elapsed=0.0 s, stab=
[4/12] WakeNumIter=10
    passed=True, elapsed=0.0 s, stab=
[5/12] WakeNumIter=11
    passed=True, elapsed=0.0 s, stab=
[6/12] WakeNumIter=13
    passed=True, elapsed=0.0 s, stab=
[7/12] WakeNumIter=14
    passed=True, elapsed=0.0 s, stab=
[8/12] WakeNumIter=15
    passed=True, elapsed=0.0 s, stab=
[9/12] WakeNumIter=16
    passed=True, elapsed=0.0 s, stab=
[10/12] WakeNumIter=18
    passed=True, elapsed=0.0 s, stab=
[11/12] WakeNumIter=19
    passed=True, elapsed=0.0 s, stab=
[12/12] WakeNumIter=20
    passed=True, elapsed=0.0 s, stab=
Done. elapsed=0.1 s
results_path: C:\Users\mtkbirdman\OneDrive\Documents\OpenVSP\examples\notebooks\wake_iter_sensitivity\wake_iter_sensitivity.csv


## 6. 結果の要約

全結果を再び解析する前に、成功可否、実行時間、Control Surface Groupの対応を確認する。


In [6]:
summary_columns = [
    "wake_iter",
    "passed",
    "elapsed_s",
    "vspaero_sweep_elapsed_s",
    "control_column_delta_e",
    "control_column_delta_a",
    "control_column_delta_r",
    "stab_path",
]

available_summary_columns = [
    column
    for column in summary_columns
    if column in results.columns
]

display(results[available_summary_columns])

failed_cases = results.loc[
    ~results["passed"].astype(bool),
    ["wake_iter", "errors", "warnings"],
]

if failed_cases.empty:
    print("All cases passed.")
else:
    print("Failed cases:")
    display(failed_cases)


,wake_iter,passed,elapsed_s,vspaero_sweep_elapsed_s,stab_path
0,3,True,0.000757,NaN,
1,4,True,0.000583,NaN,
2,8,True,0.001018,NaN,
3,10,True,0.000657,NaN,
4,11,True,0.001229,NaN,
5,13,True,0.000461,NaN,
6,14,True,0.000527,NaN,
7,15,True,0.000805,NaN,
8,16,True,0.000603,NaN,
9,18,True,0.000436,NaN,


All cases passed.


## 7. 出力ファイルの確認

各caseディレクトリに、入力`.vsp3`とVSPAEROの主要出力が存在するかを一覧で確認する。


In [7]:
file_rows = []

for wake_iter in wake_iters:
    case_name = f"wake_iter_{wake_iter:03d}"
    case_dir = output_dir / case_name

    file_rows.append(
        {
            "wake_iter": wake_iter,
            "case_dir_exists": case_dir.exists(),
            "vsp3": any(case_dir.glob("*.vsp3")),
            "vspaero": any(case_dir.glob("*.vspaero")),
            "stab": any(case_dir.glob("*.stab")),
            "adb": any(case_dir.glob("*.adb")),
            "cases": any(case_dir.glob("*.cases")),
        }
    )

display(pd.DataFrame(file_rows))


,wake_iter,case_dir_exists,vsp3,vspaero,stab,adb,cases
0,3,True,True,False,False,False,False
1,4,True,True,False,False,False,False
2,8,True,True,False,False,False,False
3,10,True,True,False,False,False,False
4,11,True,True,False,False,False,False
5,13,True,True,False,False,False,False
6,14,True,True,False,False,False,False
7,15,True,True,False,False,False,False
8,16,True,True,False,False,False,False
9,18,True,True,False,False,False,False
